# Width-distill the EDM itself (420 → 256 / 128)

FM at matched NFE was the same EGNN cost and did not beat the teacher at 8 steps.
This notebook drops the student parametrization and **compresses the production EDM**:
a thinner `EGNNDynamics` that still predicts ε and still runs through
`EquivariantDiffusion.denoise` (100 steps by default, or 8).

What is distilled:

- Teacher: frozen `edm_moi_chembl_15_39.pt` (hidden=420), production γ table `T=100`.
- Student: `EquivariantDiffusion` with `hidden_nf = WIDTH` (256, then 128).
- States: `z_t = α_t x1 + σ_t ε` on **fresh independent ε**, `t` uniform on the T=100 grid.
  `x1` comes from the existing teacher-pair shards (EDM samples, not FM).
- Loss: masked ε-MSE vs `teacher.phi` — native EDM output, no `1/α` decode, no v-field.
- Init: magnitude-ranked channel slice of a wider net, then **output heads ×0.05**.
  Raw slices explode on molecule-like states; this is the same fix as the FM width cell.
  **Do not slice 420→128 directly.** Run 256 first, then set
  `SOURCE_CKPT = "./checkpoints_edm/best_256_edm.pt"` for 128, keeping the 420 EDM as the target teacher.

First-batch loss (pre-update) should start ≈ 0.5–1, same ballpark as a cold net.
Anything ≫ 1 means the slice/head-scale is broken again. The signal to watch is descent
below that, plus the sampling cell (molecules at 100 steps, then 8).

Writes `checkpoints_edm/best_{WIDTH}_edm.pt` (`state_dict` is an `EquivariantDiffusion`).
`MLConformerGenerator` currently hardcodes `hidden_nf=420` — load this checkpoint with a matching `EGNNDynamics` until that constructor takes `hidden_nf`.

In [ ]:
"""Warm-started EDM width distillation: slice 420 -> WIDTH, match teacher.phi (ε)."""
import copy
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim import AdamW
from tqdm import tqdm

try:
    from src.mlconfgen.egnn import EGNNDynamics
    from src.mlconfgen.equivariant_diffusion import EquivariantDiffusion, PredefinedNoiseSchedule
    from src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from src.mlconfgen.utils.mol_utils import prepare_masks
except ImportError:
    from ml_conformer_generator.src.mlconfgen.egnn import EGNNDynamics
    from ml_conformer_generator.src.mlconfgen.equivariant_diffusion import (
        EquivariantDiffusion, PredefinedNoiseSchedule,
    )
    from ml_conformer_generator.src.mlconfgen.utils import CONTEXT_NORMS, MAX_N_NODES, remove_mean_with_mask
    from ml_conformer_generator.src.mlconfgen.utils.mol_utils import prepare_masks

# --------------------- config ---------------------
device = "cuda"
WIDTH = 256              # then rerun with 128
# For WIDTH=128 point this at the FINE-TUNED 256 (direct 420->128 slicing is broken):
#   SOURCE_CKPT = "./checkpoints_edm/best_256_edm.pt"
SOURCE_CKPT = None
BATCH = 64
EPOCHS = 16
LR = 1e-4
EMA_DECAY = 0.999
HEAD_SCALE = 0.05
TEACHER_T = 100          # production γ table (MLConformerGenerator default)
NOISE_PRECISION = 1e-5
PAD_TO = MAX_N_NODES
PART_CYCLE = [0, 1]
EARLY_STOP_PATIENCE = 6
MIN_DELTA = 1e-4

PAIR_DIR = Path("./teacher_pairs/teacher_pairs")
EDM_WEIGHTS = Path("./edm_moi_chembl_15_39.pt")
CKPT_DIR = Path("./checkpoints_edm")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = CKPT_DIR / f"train_{WIDTH}_edm.log"


# --------------------- boilerplate ---------------------
class EMA:
    def __init__(self, beta):
        self.beta = beta

    def update_model_average(self, ma, cur):
        for c, m in zip(cur.parameters(), ma.parameters()):
            m.data = m.data * self.beta + (1 - self.beta) * c.data


class Queue:
    def __init__(self, max_len=50):
        self.items, self.max_len = [], max_len

    def __len__(self):
        return len(self.items)

    def add(self, x):
        self.items.insert(0, x)
        if len(self) > self.max_len:
            self.items.pop()

    def mean(self):
        return float(np.mean(self.items))

    def std(self):
        return float(np.std(self.items))


def gradient_clipping(model, q):
    mx = 1.5 * q.mean() + 2.0 * q.std()
    gn = float(torch.nn.utils.clip_grad_norm_(model.parameters(), mx))
    q.add(mx if gn > mx else gn)


def log(msg):
    line = f"{datetime.now().isoformat(timespec='seconds')}  {msg}"
    print(line)
    with LOG_PATH.open("a") as f:
        f.write(line + "\n")


def com_project(z, nm):
    return torch.cat([remove_mean_with_mask(z[..., :3], nm), z[..., 3:]], -1) * nm


def load_part_pairs(part_i):
    paths = sorted(PAIR_DIR.glob(f"part_{part_i + 1}_shard*.pt")) or sorted(
        PAIR_DIR.glob(f"part{part_i + 1}_shard*.pt")
    )
    if not paths:
        paths = sorted(PAIR_DIR.glob("shard*.pt"))
        if part_i != 0:
            raise FileNotFoundError(PAIR_DIR)
    packs = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    return (
        torch.cat([p["x1"] for p in packs]),
        torch.cat([p["n_atoms"] for p in packs]),
        torch.cat([p["context"] for p in packs]),
    )


def eps_loss(pred, tgt, nm):
    err = (pred - tgt) ** 2 * nm
    n = nm.sum().clamp_min(1)
    lx = err[..., :3].sum() / (n * 3)
    lh = err[..., 3:].sum() / (n * 8)
    return lx + lh, lx.detach(), lh.detach()


def n_params(m):
    return sum(p.numel() for p in m.parameters())


def make_edm(hidden_nf):
    dyn = EGNNDynamics(in_node_nf=9, context_node_nf=3, hidden_nf=hidden_nf, device=device)
    mdl = EquivariantDiffusion(dyn, in_node_nf=8, timesteps=1000, noise_precision=NOISE_PRECISION)
    mdl.gamma = PredefinedNoiseSchedule(timesteps=TEACHER_T, precision=NOISE_PRECISION)
    mdl.time_steps = torch.flip(torch.arange(0, TEACHER_T, device=device), dims=[0])
    mdl.T = TEACHER_T
    return mdl.to(device)


def set_steps(mdl, n):
    mdl.gamma = PredefinedNoiseSchedule(timesteps=n, precision=NOISE_PRECISION)
    mdl.time_steps = torch.flip(torch.arange(0, n, device=device), dims=[0])
    mdl.T = n


# --------------------- warm start: slice wider EGNN -> WIDTH ---------------------
def _row_norm(W):
    return W.pow(2).sum(1)


def _col_norm(W):
    return W.pow(2).sum(0)


def _topk(score, k):
    return torch.topk(score, k).indices.sort().values


@torch.no_grad()
def warm_start_from(big_dyn, small_dyn, Hb, Hs, head_scale=HEAD_SCALE):
    """Magnitude-ranked channel slice, then shrink output heads (see intro)."""
    bg, sm = big_dyn.egnn, small_dyn.egnn
    blocks_b = [getattr(bg, f"e_block_{i}") for i in range(9)]
    blocks_s = [getattr(sm, f"e_block_{i}") for i in range(9)]
    dev = bg.embedding.weight.device

    score = _row_norm(bg.embedding.weight) + _col_norm(bg.embedding_out.weight)
    for blk in blocks_b:
        for g in (blk.gcl_0, blk.gcl_1):
            W0 = g.edge_mlp[0].weight
            score += _col_norm(W0[:, :Hb]) + _col_norm(W0[:, Hb:2 * Hb])
            score += _col_norm(g.node_mlp[0].weight[:, :Hb])
            score += _row_norm(g.node_mlp[2].weight)
        Wc = blk.gcl_equiv.coord_mlp[0].weight
        score += _col_norm(Wc[:, :Hb]) + _col_norm(Wc[:, Hb:2 * Hb])
    S = _topk(score, Hs)

    def cp(dst, W, b=None):
        dst.weight.data.copy_(W)
        if b is not None and dst.bias is not None:
            dst.bias.data.copy_(b)

    tail2 = torch.tensor([2 * Hb, 2 * Hb + 1], device=dev)
    pair_cols = torch.cat([S, S + Hb, tail2])

    cp(sm.embedding, bg.embedding.weight[S], bg.embedding.bias[S])
    cp(sm.embedding_out, bg.embedding_out.weight[:, S], bg.embedding_out.bias)

    for blk_b, blk_s in zip(blocks_b, blocks_s):
        for gb, gs in ((blk_b.gcl_0, blk_s.gcl_0), (blk_b.gcl_1, blk_s.gcl_1)):
            W0, b0 = gb.edge_mlp[0].weight, gb.edge_mlp[0].bias
            W2, b2 = gb.edge_mlp[2].weight, gb.edge_mlp[2].bias
            Wa, ba = gb.att_mlp[0].weight, gb.att_mlp[0].bias
            Wn0, bn0 = gb.node_mlp[0].weight, gb.node_mlp[0].bias
            Wn2, bn2 = gb.node_mlp[2].weight, gb.node_mlp[2].bias

            E = _topk(_row_norm(W0) + _col_norm(W2), Hs)
            M = _topk(_row_norm(W2) + _col_norm(Wa) + _col_norm(Wn0[:, Hb:]), Hs)
            N = _topk(_row_norm(Wn0) + _col_norm(Wn2), Hs)

            cp(gs.edge_mlp[0], W0[E][:, pair_cols], b0[E])
            cp(gs.edge_mlp[2], W2[M][:, E], b2[M])
            cp(gs.att_mlp[0], Wa[:, M], ba)
            node_cols = torch.cat([S, M + Hb])
            cp(gs.node_mlp[0], Wn0[N][:, node_cols], bn0[N])
            cp(gs.node_mlp[2], Wn2[S][:, N], bn2[S])

        cb, cs = blk_b.gcl_equiv.coord_mlp, blk_s.gcl_equiv.coord_mlp
        W0, b0 = cb[0].weight, cb[0].bias
        W2, b2 = cb[2].weight, cb[2].bias
        W4 = cb[4].weight
        C1 = _topk(_row_norm(W0) + _col_norm(W2), Hs)
        C2 = _topk(_row_norm(W2) + _col_norm(W4), Hs)
        cp(cs[0], W0[C1][:, pair_cols], b0[C1])
        cp(cs[2], W2[C2][:, C1], b2[C2])
        cs[4].weight.data.copy_(W4[:, C2])

    sm.embedding_out.weight.data.mul_(head_scale)
    sm.embedding_out.bias.data.mul_(head_scale)
    for blk in blocks_s:
        blk.gcl_equiv.coord_mlp[4].weight.data.mul_(head_scale)
    return S


# --------------------- models ---------------------
edm_ckpt = torch.load(EDM_WEIGHTS, map_location="cpu", weights_only=False)
norms = {
    k: torch.tensor(v, device=device, dtype=torch.float32)
    for k, v in edm_ckpt.get("context_norms", CONTEXT_NORMS).items()
}
HB = 420
teacher = make_edm(HB)
teacher.load_state_dict(edm_ckpt["state_dict"])
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)

student = make_edm(WIDTH)
if SOURCE_CKPT is None:
    src_model, HS, src_name = teacher, HB, EDM_WEIGHTS.name
else:
    sck = torch.load(SOURCE_CKPT, map_location=device, weights_only=False)
    HS, src_name = int(sck["hidden_nf"]), Path(SOURCE_CKPT).name
    src_model = make_edm(HS)
    src_model.load_state_dict(sck.get("student") or sck["state_dict"])
assert HS > WIDTH, f"source width {HS} must exceed student width {WIDTH}"
warm_start_from(src_model.dynamics, student.dynamics, HS, WIDTH)
log(
    f"warm-started {HS}->{WIDTH} from {src_name} "
    f"(keep {WIDTH / HS:.0%} of channels, head_scale={HEAD_SCALE})"
)
log(f"params teacher={n_params(teacher):,}  student={n_params(student):,}")
if src_model is not teacher:
    del src_model
    torch.cuda.empty_cache()

student_ema = copy.deepcopy(student).to(device)
for p in student_ema.parameters():
    p.requires_grad_(False)
ema = EMA(EMA_DECAY)
opt = AdamW(student.parameters(), lr=LR, amsgrad=True, weight_decay=1e-12)
q = Queue()
q.add(100.0)

log(f"start EDM width-distill {HB}->{WIDTH} T={TEACHER_T} batch={BATCH}")
best, no_imp, logged_first = float("inf"), 0, False

for epoch in range(EPOCHS):
    part_i = PART_CYCLE[epoch % len(PART_CYCLE)]
    x1_all, na_all, ctx_all = load_part_pairs(part_i)
    perm = torch.randperm(x1_all.shape[0])
    x1_all, na_all, ctx_all = x1_all[perm], na_all[perm], ctx_all[perm]
    student.train()
    run, rx, rh, ns = 0.0, 0.0, 0.0, 0
    pbar = tqdm(range(0, x1_all.shape[0] - BATCH + 1, BATCH), desc=f"edm{WIDTH} ep{epoch}")
    for i in pbar:
        sl = slice(i, i + BATCH)
        na = na_all[sl].to(device=device, dtype=torch.long)
        nm, em = prepare_masks(na, PAD_TO, device)
        ctx = ctx_all[sl].to(device=device, dtype=torch.float32)
        bctx = ((ctx - norms["mean"]) / norms["mad"]).unsqueeze(1).expand(-1, PAD_TO, -1) * nm
        x1_b = com_project(x1_all[sl].to(device=device, dtype=torch.float32), nm)
        B, N = x1_b.shape[:2]

        t = torch.randint(0, TEACHER_T, (B, 1), device=device).float() / TEACHER_T
        eps = teacher.sample_combined_position_feature_noise(B, N, nm)
        gamma = teacher.gamma(t)
        a_t = teacher.alpha(gamma, x1_b)
        s_t = teacher.sigma(gamma, x1_b)
        zt = com_project(a_t * x1_b + s_t * eps, nm)

        with torch.no_grad():
            eps_tgt = teacher.phi(zt, t, nm, em, bctx)
        eps_pred = student.phi(zt, t, nm, em, bctx)
        loss, lx, lh = eps_loss(eps_pred, eps_tgt, nm)
        if not logged_first:
            log(
                f"first-batch loss (pre-update, = warm-start quality): {loss.item():.4f} "
                f"(should be ~0.5–1; ≫ 1 means heads were not scaled)"
            )
            logged_first = True
        if not torch.isfinite(loss):
            log(f"WARN non-finite ep={epoch}")
            continue
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gradient_clipping(student, q)
        opt.step()
        ema.update_model_average(student_ema, student)
        run += loss.item(); rx += lx.item(); rh += lh.item(); ns += 1
        if ns % 20 == 0:
            pbar.set_postfix(loss=f"{run/ns:.5f}", lx=f"{rx/ns:.5f}", lh=f"{rh/ns:.5f}")

    avg = run / max(ns, 1)
    log(f"epoch={epoch} avg={avg:.6f} lx={rx/max(ns,1):.6f} lh={rh/max(ns,1):.6f}")
    ckpt = {
        "epoch": epoch, "avg_loss": avg, "stage": f"edm{WIDTH}",
        "hidden_nf": WIDTH, "T": TEACHER_T, "param": "eps",
        "teacher": EDM_WEIGHTS.name, "warm_start": True,
        "state_dict": student.state_dict(),
        "student": student.state_dict(),
        "student_ema": student_ema.state_dict(),
        "context_norms": {k: v.detach().cpu().tolist() for k, v in norms.items()},
        "opt": opt.state_dict(),
    }
    torch.save(ckpt, CKPT_DIR / f"latest_{WIDTH}_edm.pt")
    if avg < best - MIN_DELTA:
        best, no_imp = avg, 0
        torch.save(ckpt, CKPT_DIR / f"best_{WIDTH}_edm.pt")
        log(f"ckpt best avg={avg:.6f}")
    else:
        no_imp += 1
        best = min(best, avg)
    if no_imp >= EARLY_STOP_PATIENCE:
        log(f"early stop epoch={epoch}")
        break

log(f"done EDM width={WIDTH} best={best:.6f}")

In [ ]:
"""Gate: teacher EDM vs this student, ancestral sampling at 100 and 8 steps."""
import math
import time

import py3Dmol
from rdkit import Chem

try:
    from src.mlconfgen.utils import ATOM_DECODER, align_mol_to_principal_frame, prepare_edm_input, samples_to_rdkit_mol
except ImportError:
    from ml_conformer_generator.src.mlconfgen.utils import (
        ATOM_DECODER, align_mol_to_principal_frame, prepare_edm_input, samples_to_rdkit_mol,
    )

MOL_PATH = "./assets/demo_files/ceyyag.mol"
N_SHOW = 4
STEPS = (100, 8)
torch.manual_seed(42)


def show(ms):
    blocks = [Chem.MolToXYZBlock(m) for m in ms if m is not None]
    cols = min(4, len(blocks))
    rows = math.ceil(len(blocks) / cols)
    v = py3Dmol.view(viewergrid=(rows, cols), width=250 * cols, height=250 * rows)
    for i, b in enumerate(blocks):
        r, c = divmod(i, cols)
        v.addModel(b, "xyz", viewer=(r, c))
        v.setStyle({"stick": {"radius": 0.15}, "sphere": {"scale": 0.25}}, viewer=(r, c))
        v.zoomTo(viewer=(r, c))
    v.show()


@torch.inference_mode()
def sample_edm(model, nm, em, ctx, n_steps, tag):
    set_steps(model, n_steps)
    model.eval()
    t0 = time.perf_counter()
    x, h = model(nm, em, ctx)
    dt = time.perf_counter() - t0
    finite = torch.isfinite(x).all()
    std = x[nm[..., 0].bool()].std().item() if finite else float("nan")
    print(f"{tag} NFE={n_steps}: {dt*1000/nm.shape[0]:.1f} ms/mol  coord std={std:.3f}  finite={bool(finite)}")
    return x, h


ref = Chem.RemoveAllHs(Chem.MolFromMolFile(MOL_PATH))
ctx3, *_ = align_mol_to_principal_frame(ref)
na_ref = ref.GetNumAtoms()
nm_s, em_s, ctx_s = prepare_edm_input(
    N_SHOW, ctx3.to(device), norms, na_ref, na_ref, device, pad_to=PAD_TO
)

for n_steps in STEPS:
    print(f"\n=== ancestral {n_steps} steps ===")
    xt, ht = sample_edm(teacher, nm_s, em_s, ctx_s, n_steps, "teacher420")
    show(samples_to_rdkit_mol(xt.cpu(), ht.cpu(), nm_s.cpu(), ATOM_DECODER))
    xs, hs = sample_edm(student, nm_s, em_s, ctx_s, n_steps, f"student{WIDTH}")
    show(samples_to_rdkit_mol(xs.cpu(), hs.cpu(), nm_s.cpu(), ATOM_DECODER))